# Data validation and cleaning

In [22]:
import sys
from pathlib import Path
import pandas as pd 

# adds the root carpet of the project to the sys.path to allow importing modules from src
sys.path.append(str(Path.cwd().parent))

# autoreload configuration to automatically reload modules when they are modified
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Auditoría de Datos (Pruebas Individuales)

In [23]:
from src.data.validate_data import describe_yfinance_data
from src.data.validate_data import summarize_date_gaps
from src.data.validate_data import count_nans_by_ticker
from src.data.validate_data import has_duplicate_dates
from src.data.validate_data import is_index_sorted
from src.data.validate_data import detect_impossible_values
from src.data.validate_data import get_trading_periods_by_ticker
from src.data.validate_data import check_yfinance_dtypes

from src.data.validate_data import create_data_quality_summary

### 1.1 Análisis de la información general (frecuencia, fechas y observaciones)

In [24]:
prices = pd.read_parquet("../data/raw/sp500_prices.parquet")

info = describe_yfinance_data(prices)
print(info)

{'n_assets': 499, 'n_observations': 3773, 'start_date': Timestamp('2010-01-04 00:00:00'), 'end_date': Timestamp('2024-12-30 00:00:00')}


Justamente el número de empresas que tenemos es la diferencia entre el total (las 503 empresas) y las 5 empresas que nos dieron problemas (eso se puede ver en el jupyter de data_extraction). 

El número de observaciones corresponde al número de días en que los mercados han estado abiertos en el intervalo temporal que hemos escogido. 

El start date y el end date son correctos. 

In [25]:
date_gaps = summarize_date_gaps(prices)
print(date_gaps)

1 day     2955
2 days      35
3 days     680
4 days     101
5 days       1
Name: count, dtype: int64


Para saber si los datos obtenidos son realmente diarios habría sido un problema definir una función que te diese True si "todos los datos fuesen diarios" ya que obviamente los mercados no abren todos los días, existen festivos, findes de semana, etc. En lugar de eso hemos optado por encontrar el número y el tipo de saltos temporales entre días consecutivos. 

Si analizamos el resultado vemos que la gran mayoría de saltos son de 1 día (es decir, cuando dos días hábiles seguidos son también días naturales), luego tenemos un pico en los 3 días (que corresponde a los findes de semana) y solo hay un salto de más de 4 días. Realizando este análisis nos queda claro que los datos son diarios.  

### 1.2 Análisis de los valores NaN y trading periods

In [26]:
nan_counts = count_nans_by_ticker(prices)
display(nan_counts)

,total_nans,internal_nans,nans_outside_trading_period
Ticker,,,
GEV,21486,0,21486
SOLV,21480,0,21480
VLTO,20766,0,20766
KVUE,20136,0,20136
GEHC,19566,0,19566
...,...,...,...
AES,0,0,0
AFL,0,0,0
AIG,0,0,0


Gracias a esta función nos hacemos una idea de cuántos valores NaN estamos manejando en total (sumando todas las variables) respecto a cada una de las empresas. Esta información será útil a la hora de decidir si introducimos solamente las empresas con una cotización suficientemente extensa, etc. 

Además nos damos cuenta de que los NaN que aparecen no son internos, es decir, son el resultado de que no todas las empresas comenzaron a cotizar en 2010 (y a lo mejor no todas las empresas han llegado hasta el 2024). Este tipo de NaN es el que llamamos nans_outside_trading_period. 

Esta información es importante a la hora de construir los factores ya que nos quitamos del problema de tener días en medio  en los que nos comemos un valor nulo.

In [27]:
trading_periods = get_trading_periods_by_ticker(prices)

full_period_tickers = trading_periods["full_period"]
partial_period_tickers = trading_periods["partial_period"]

print(full_period_tickers)
print(partial_period_tickers)

       first_trading_date last_trading_date  traded_full_period
Ticker                                                         
A              2010-01-04        2024-12-30                True
AAPL           2010-01-04        2024-12-30                True
ABT            2010-01-04        2024-12-30                True
ACGL           2010-01-04        2024-12-30                True
ACN            2010-01-04        2024-12-30                True
...                   ...               ...                 ...
XEL            2010-01-04        2024-12-30                True
XOM            2010-01-04        2024-12-30                True
YUM            2010-01-04        2024-12-30                True
ZBH            2010-01-04        2024-12-30                True
ZBRA           2010-01-04        2024-12-30                True

[421 rows x 3 columns]
       first_trading_date last_trading_date  traded_full_period
Ticker                                                         
ABBV           2

Obtenemos 421 empresas con sus correspondientes tickers que han cotizado durante todo el intervalo propuesto (enero 2010 a diciembre 2024). Por otro lado tenemos 78 empresas que no han cotizado durante todo ese tiempo y el 100% de ellas se debe a que han comenzado a cotizar después del 2010 (es decir, que todas llegan a cotizar hasta el último día de 2024). 

### 1.3 Análisis de estructura

Aquí cubriremos las restricciones de precios negativos y otros valores imposibles, el correcto orden cronológico y de índices y los data types espcíficos en cada caso

In [28]:
print(has_duplicate_dates(prices))  # True if duplicated dates exist
print(is_index_sorted(prices))      # True if the index is chronologically sorted

False
True


Observamos que en nuestro dataset tenemos todas las fechas puestas en su correcto orden cronológico y no encontramos fechas duplicadas. 

In [29]:
invalid_values = detect_impossible_values(prices)

display(invalid_values)

display(invalid_values[invalid_values['zero_volume'] != 0].tail())

n_companies_with_zero_volume = (invalid_values["zero_volume"] > 0).sum()

print(f"Number of companies with at least one zero volume day: {n_companies_with_zero_volume}")

,invalid_open,invalid_high,invalid_low,invalid_close,invalid_adj_close,zero_volume,negative_volume,low_greater_than_high,high_less_than_open,high_less_than_close,low_greater_than_open,low_greater_than_close,total_issues
Ticker,,,,,,,,,,,,,
SW,0,0,0,0,0,2409,0,0,0,0,0,0,2409
AMCR,0,0,0,0,0,1371,0,0,0,0,0,0,1371
HWM,0,0,0,0,0,35,0,0,0,0,0,0,35
CHTR,0,0,0,0,0,18,0,0,0,0,0,0,18
VRT,0,0,0,0,0,12,0,0,0,0,0,0,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...
AES,0,0,0,0,0,0,0,0,0,0,0,0,0
AFL,0,0,0,0,0,0,0,0,0,0,0,0,0
AIG,0,0,0,0,0,0,0,0,0,0,0,0,0


,invalid_open,invalid_high,invalid_low,invalid_close,invalid_adj_close,zero_volume,negative_volume,low_greater_than_high,high_less_than_open,high_less_than_close,low_greater_than_open,low_greater_than_close,total_issues
Ticker,,,,,,,,,,,,,
ERIE,0,0,0,0,0,1,0,0,0,0,0,0,1
DOC,0,0,0,0,0,1,0,0,0,0,0,0,1
SBAC,0,0,0,0,0,1,0,0,0,0,0,0,1
WTW,0,0,0,0,0,1,0,0,0,0,0,0,1
XEL,0,0,0,0,0,1,0,0,0,0,0,0,1


Number of companies with at least one zero volume day: 23


La primera tabla nos resume la información general y la segunda la hemos puesto para poder ver las empresas con menor número de días (quitando los cero días) con volumen cero. En total nos salen 23 empresas con un volumen nulo en alguno de sus días. 

Las razones principales para encontrar un volumen cero son las suspensiones temporales de cotización por parte de la bolsa debido a noticias o volatilidad, la falta total de liquidez en empresas muy pequeñas donde no se cruza ninguna operación en todo el día, o la presencia de festivos locales y cierres parciales de mercado. Por otro lado, hay que tener mucho cuidado con estos datos porque en una simulación o backtest un algoritmo podría intentar ejecutar compras o ventas ficticias que en la realidad habrían sido imposibles por falta de contrapartida; además, el precio suele mantenerse congelado respecto al día anterior dando una falsa sensación de estabilidad, y aunque borrar estas filas parece la solución fácil, hacerlo rompería la alineación temporal de todo el panel de datos, por lo que lo correcto es conservar el dato pero filtrar o restringir la operativa sobre esa acción durante la fase de selección del universo.

Por otro lado encontramos que ninguna de las empresas tiene ningún valor "imposible" en ninguna de las variables ni combinaciones de estas. 

In [30]:
dtype_report = check_yfinance_dtypes(prices)
print(dtype_report)

          expected_dtype    actual_dtype  is_valid
field                                             
Date          datetime64  datetime64[ms]      True
Open             float64         float64      True
High             float64         float64      True
Low              float64         float64      True
Close            float64         float64      True
Volume  int64 or float64  int64, float64      True


Por último, comprobamos que todos los data types son los adecuados en cada caso. 

### 1.4 Resumen del análisis

A modo de resumen, introducimos una tabla con la información más relevante que hemos ido viendo. 

In [31]:
quality_summary = create_data_quality_summary(prices)
quality_summary

,metric,value
0,Number of assets,499
1,Number of observations,3773
2,Start date,2010-01-04 00:00:00
3,End date,2024-12-30 00:00:00
4,Duplicated dates,False
5,Chronologically sorted index,True
6,Tickers with data for the full period,421
7,Tickers with partial trading periods,78


## 2. Limpieza y Transformación



### 2.1 Conclusiones de la fase de *Data Cleaning*

Tras completar la fase de validación, se concluye que el conjunto de datos no requiere un proceso de limpieza adicional antes de continuar con el pipeline.

Las comprobaciones realizadas muestran que:

* No existen fechas duplicadas y el índice temporal se encuentra correctamente ordenado.
* No se han detectado valores imposibles en los precios (precios negativos o inconsistencias entre *High*, *Low*, *Open* y *Close*).
* No existen volúmenes negativos.
* Los tipos de datos son los esperados para todas las variables.
* No se han encontrado valores perdidos (*missing values*) durante el periodo de cotización de ninguna empresa.

Los únicos valores `NaN` presentes corresponden a empresas que comenzaron a cotizar después del 1 de enero de 2010. Estos valores no representan errores ni datos faltantes, sino la ausencia legítima de información histórica previa a la salida a bolsa (*IPO*). Por tanto, se conservarán en el conjunto de datos y serán tratados adecuadamente durante la construcción de factores y modelos, respetando el universo invertible disponible en cada instante temporal.

Asimismo, se han identificado algunos casos aislados de volumen igual a cero. Dado que su origen puede responder a suspensiones temporales de cotización, ausencia de operaciones o particularidades del proveedor de datos, se ha decidido no imputar ni modificar estos valores en esta fase. Cualquier tratamiento específico se realizará únicamente cuando se construyan factores que dependan del volumen, evitando introducir supuestos artificiales en el conjunto de datos original.

En consecuencia, el dataset obtenido tras la validación se considera suficientemente consistente para utilizarse como conjunto de datos de referencia durante el resto del proyecto. El pipeline continúa, por tanto, con el dataset validado sin aplicar transformaciones adicionales de limpieza, manteniendo la máxima fidelidad posible respecto a los datos originales descargados.
